# Custom Kafka Generators (Experimental)

Define ad-hoc Kafka streaming generators via JSON column specs — no server code changes needed.

The column spec maps to [dbldatagen's](https://github.com/databrickslabs/dbldatagen) `withColumn()` API.

**Column spec options:**

| Option | Description | Example |
|---|---|---|
| `expr` | Spark SQL expression | `"uuid()"`, `"rand() < 0.3"`, `"concat('ID-', ...)"` |
| `values` / `weights` | Categorical with optional weights | `["A","B","C"]` / `[5,3,2]` |
| `min_value` / `max_value` | Numeric range | `0.0` / `100.0` |
| `random` | Randomize (vs sequential) | `true` |
| `template` | String template | `"user_\\0"` |
| `unique_values` | Distinct value count | `1000` |
| `percent_nulls` | Null percentage (0-100) | `5.0` |
| `begin` / `end` | Date/timestamp range | `"2024-01-01"` |
| `base_column` | Derive from another column | `"user_id"` |

In [ ]:
%run "./_config"

---
## Example 1: IoT Sensor Stream

Simulate IoT devices reporting temperature, humidity, and battery levels.

In [ ]:
iot_spec = {
    "name": "iot_sensors",
    "topic_name": "custom-iot-sensors",
    "rows_per_batch": 50,
    "batch_interval_seconds": 2.0,
    "timeout_minutes": 10,
    "columns": [
        {"name": "device_id", "type": "string",
         "expr": "concat('DEV-', lpad(cast(int(rand()*500+1) as string), 4, '0'))"},
        {"name": "device_type", "type": "string",
         "values": ["temperature", "humidity", "pressure", "air_quality", "motion"],
         "random": True, "weights": [4, 3, 2, 2, 1]},
        {"name": "reading", "type": "decimal(8,3)",
         "min_value": -40.0, "max_value": 150.0, "random": True},
        {"name": "unit", "type": "string",
         "values": ["celsius", "percent", "hpa", "ppm", "boolean"], "random": True},
        {"name": "battery_pct", "type": "integer",
         "min_value": 0, "max_value": 100, "random": True},
        {"name": "signal_dbm", "type": "integer",
         "min_value": -120, "max_value": -20, "random": True},
        {"name": "facility", "type": "string",
         "values": ["Building-A", "Building-B", "Warehouse-1", "Warehouse-2", "Field-Site"],
         "random": True},
        {"name": "anomaly", "type": "boolean", "expr": "rand() < 0.05"},
        {"name": "latitude", "type": "decimal(8,5)",
         "min_value": 38.0, "max_value": 42.0, "random": True},
        {"name": "longitude", "type": "decimal(9,5)",
         "min_value": -78.0, "max_value": -73.0, "random": True},
    ],
}

In [ ]:
# Validate first — see the resolved schema and a sample row
validation = requests.post(
    f"{API_URL}/kafka/generators/custom/validate",
    headers=HEADERS, json=iot_spec,
).json()
print("Schema:", validation.get("resolved_schema"))
print("Sample:", validation.get("sample_row"))

In [ ]:
# Start the generator
resp = requests.post(
    f"{API_URL}/kafka/generators/custom/start",
    headers=HEADERS, json=iot_spec,
).json()
iot_id = resp["generator_id"]
print(resp)

In [ ]:
# Read the custom stream
iot_schema = StructType([
    StructField("device_id", StringType()),
    StructField("device_type", StringType()),
    StructField("reading", DecimalType(8, 3)),
    StructField("unit", StringType()),
    StructField("battery_pct", IntegerType()),
    StructField("signal_dbm", IntegerType()),
    StructField("facility", StringType()),
    StructField("anomaly", BooleanType()),
    StructField("latitude", DecimalType(8, 5)),
    StructField("longitude", DecimalType(9, 5)),
    StructField("event_timestamp", StringType()),
])

iot_df = read_kafka_stream("custom-iot-sensors", iot_schema)

dbutils.fs.rm(f"{CHECKPOINT_BASE}/custom_iot_preview", recurse=True)
display(iot_df, checkpointLocation=f"{CHECKPOINT_BASE}/custom_iot_preview")


---
## Example 2: E-Commerce Clickstream

Simulate user sessions browsing an e-commerce site.

In [ ]:
clickstream_spec = {
    "name": "ecommerce_clicks",
    "topic_name": "custom-clickstream",
    "rows_per_batch": 200,
    "batch_interval_seconds": 0.5,
    "timeout_minutes": 10,
    "columns": [
        {"name": "click_id", "type": "string", "expr": "uuid()"},
        {"name": "session_id", "type": "string",
         "expr": "concat('sess-', lpad(cast(int(rand()*50000+1) as string), 6, '0'))"},
        {"name": "user_id", "type": "integer", "min_value": 1, "max_value": 100000, "random": True},
        {"name": "page", "type": "string",
         "values": ["/home", "/products", "/product/detail", "/cart", "/checkout",
                    "/search", "/account", "/wishlist", "/reviews", "/support"],
         "random": True, "weights": [5, 4, 3, 2, 1, 3, 1, 1, 1, 1]},
        {"name": "action", "type": "string",
         "values": ["view", "click", "scroll", "add_to_cart", "remove_from_cart",
                    "purchase", "search", "filter", "sort"],
         "random": True, "weights": [5, 3, 2, 2, 1, 1, 2, 1, 1]},
        {"name": "product_category", "type": "string",
         "values": ["electronics", "clothing", "home", "books", "sports",
                    "beauty", "food", "toys", "automotive"],
         "random": True},
        {"name": "price", "type": "decimal(8,2)", "min_value": 0.99, "max_value": 2999.99, "random": True},
        {"name": "device", "type": "string",
         "values": ["mobile", "desktop", "tablet"], "random": True, "weights": [5, 4, 1]},
        {"name": "referrer", "type": "string",
         "values": ["direct", "google", "facebook", "instagram", "email", "affiliate"],
         "random": True, "weights": [3, 4, 2, 2, 2, 1]},
        {"name": "duration_ms", "type": "integer", "min_value": 100, "max_value": 60000, "random": True},
    ],
}

resp = requests.post(
    f"{API_URL}/kafka/generators/custom/start",
    headers=HEADERS, json=clickstream_spec,
).json()
click_id = resp["generator_id"]
print(resp)

In [ ]:
# Read the clickstream
click_schema = StructType([
    StructField("click_id", StringType()),
    StructField("session_id", StringType()),
    StructField("user_id", IntegerType()),
    StructField("page", StringType()),
    StructField("action", StringType()),
    StructField("product_category", StringType()),
    StructField("price", DecimalType(8, 2)),
    StructField("device", StringType()),
    StructField("referrer", StringType()),
    StructField("duration_ms", IntegerType()),
    StructField("event_timestamp", StringType()),
])

click_df = read_kafka_stream("custom-clickstream", click_schema)

dbutils.fs.rm(f"{CHECKPOINT_BASE}/custom_click_preview", recurse=True)
display(click_df, checkpointLocation=f"{CHECKPOINT_BASE}/custom_click_preview")


---
## Example 3: Healthcare Vitals Stream

Simulate patient vitals monitoring with nulls and anomaly flags.

In [ ]:
vitals_spec = {
    "name": "patient_vitals",
    "topic_name": "custom-patient-vitals",
    "rows_per_batch": 30,
    "batch_interval_seconds": 3.0,
    "timeout_minutes": 10,
    "columns": [
        {"name": "reading_id", "type": "string", "expr": "uuid()"},
        {"name": "patient_id", "type": "string",
         "expr": "concat('PAT-', lpad(cast(int(rand()*2000+1) as string), 5, '0'))"},
        {"name": "ward", "type": "string",
         "values": ["ICU", "ER", "General", "Pediatrics", "Cardiology", "Oncology"],
         "random": True, "weights": [2, 3, 5, 2, 2, 1]},
        {"name": "heart_rate", "type": "integer", "min_value": 40, "max_value": 180, "random": True},
        {"name": "systolic_bp", "type": "integer", "min_value": 80, "max_value": 200, "random": True},
        {"name": "diastolic_bp", "type": "integer", "min_value": 50, "max_value": 130, "random": True},
        {"name": "temperature_f", "type": "decimal(4,1)", "min_value": 95.0, "max_value": 105.0, "random": True},
        {"name": "spo2", "type": "integer", "min_value": 85, "max_value": 100, "random": True},
        {"name": "respiratory_rate", "type": "integer", "min_value": 8, "max_value": 40, "random": True},
        {"name": "pain_level", "type": "integer", "values": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
         "random": True, "weights": [10, 5, 5, 4, 4, 3, 3, 2, 2, 1, 1]},
        {"name": "alert", "type": "boolean", "expr": "rand() < 0.08"},
        {"name": "nurse_id", "type": "string",
         "expr": "concat('RN-', lpad(cast(int(rand()*200+1) as string), 3, '0'))"},
    ],
}

resp = requests.post(
    f"{API_URL}/kafka/generators/custom/start",
    headers=HEADERS, json=vitals_spec,
).json()
vitals_id = resp["generator_id"]
print(resp)

In [ ]:
vitals_schema = StructType([
    StructField("reading_id", StringType()),
    StructField("patient_id", StringType()),
    StructField("ward", StringType()),
    StructField("heart_rate", IntegerType()),
    StructField("systolic_bp", IntegerType()),
    StructField("diastolic_bp", IntegerType()),
    StructField("temperature_f", DecimalType(4, 1)),
    StructField("spo2", IntegerType()),
    StructField("respiratory_rate", IntegerType()),
    StructField("pain_level", IntegerType()),
    StructField("alert", BooleanType()),
    StructField("nurse_id", StringType()),
    StructField("event_timestamp", StringType()),
])

vitals_df = read_kafka_stream("custom-patient-vitals", vitals_schema)

dbutils.fs.rm(f"{CHECKPOINT_BASE}/custom_vitals_preview", recurse=True)
display(vitals_df, checkpointLocation=f"{CHECKPOINT_BASE}/custom_vitals_preview")


---
## Management — List, Inspect, Stop

In [ ]:
# List all custom generators
custom_gens = requests.get(f"{API_URL}/kafka/generators/custom", headers=HEADERS).json()
for g in custom_gens:
    print(f"  {g['name']} ({g['generator_id']}): {g['status']} — {g['rows_produced']} rows, {g['columns']} columns")

In [ ]:
# Get the spec back for any generator (useful for cloning / debugging)
if custom_gens:
    first_id = custom_gens[0]["generator_id"]
    spec = requests.get(f"{API_URL}/kafka/generators/custom/{first_id}/spec", headers=HEADERS).json()
    print(f"Generator: {spec['name']}")
    print(f"Topic: {spec['topic_name']}")
    print(f"Columns: {len(spec['columns'])}")
    for col in spec["columns"]:
        print(f"  - {col['name']} ({col['type']})")

In [ ]:
# Write the IoT stream to a Delta table
# (
#     iot_df.writeStream
#     .format("delta")
#     .outputMode("append")
#     .option("checkpointLocation", f"{CHECKPOINT_BASE}/custom_iot_delta")
#     .toTable(f"{DELTA_BASE}.custom_iot_sensors")
# )

---
## Cleanup

In [ ]:
# Stop all custom generators
custom_gens = requests.get(f"{API_URL}/kafka/generators/custom", headers=HEADERS).json()
for g in custom_gens:
    if g["status"] == "running":
        requests.post(f"{API_URL}/kafka/generators/custom/{g['generator_id']}/stop", headers=HEADERS)
        print(f"Stopped {g['name']} ({g['generator_id']})")

cleanup = requests.delete(f"{API_URL}/kafka/generators/custom/cleanup", headers=HEADERS).json()
print(f"Cleaned up {cleanup.get('removed', 0)} generator(s)")